# **FOREWORD**

This is a day-1 starter kernel for the community competition [Stock Market Signal: Predict Next-Day Returns](https://www.kaggle.com/competitions/stock-market-signal-predict-next-day-returns). This is a time series classifier with a low signal-noise ratio. AUC metric is the eval-metric for the assignment and needs to be maximized. 

This is a binary classifier that needs to predict the direction of the stock price for the provided stocks for the next trading date. 

This looks like a shake-up competition, so I won't be investing efforts into this. A simple starter is sufficient for this competition. 

This kernel uses a simple VotingClassifier with 4 base tree models for starters.

# **IMPORTS**

In [1]:
!uv pip install scikit-learn==1.7.2 xgboost --system -q

import pandas as pd, numpy as np
from tqdm.auto import tqdm
from warnings import filterwarnings
from gc import collect 
import os, sys, joblib, re

from sklearn.metrics import *
from sklearn.preprocessing import *
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.compose import ColumnTransformer, make_column_selector
from sklearn.base import clone, BaseEstimator, TransformerMixin, ClassifierMixin
from sklearn.impute import SimpleImputer

from xgboost import XGBClassifier as XGBC
from catboost import CatBoostClassifier as CBC
from lightgbm import LGBMClassifier as LGBMC, log_evaluation, early_stopping
from sklearn.ensemble import VotingClassifier

filterwarnings("ignore")

In [2]:
target = "target"

# **PREPROCESSING**

In [3]:
train = pd.read_csv(
    f"/kaggle/input/stock-market-signal-predict-next-day-returns/train.csv",
    index_col = "id",
)

test = pd.read_csv(
    f"/kaggle/input/stock-market-signal-predict-next-day-returns/test.csv",
    index_col = "id",
)

sub_fl = pd.read_csv(
    f"/kaggle/input/stock-market-signal-predict-next-day-returns/sample_submission.csv",
    index_col = "id",
)

print(f"---> Shapes = {train.shape} {test.shape} {sub_fl.shape}")

with np.printoptions(linewidth = 100):
    sel_cols = test.columns.tolist()[1:]
    print(f"\n---> Columns at start\n")
    print(np.array(sel_cols))

print()
display(
    pd.concat([train[sel_cols].isna().mean(), test.isna().mean()], axis=1)
    .rename(columns = {0 : "Train", 1 : "Test"})
    .transpose()
    .style
    .set_caption("Null values across columns")
)

---> Shapes = (440402, 29) (53276, 28) (53276, 1)

---> Columns at start

['return_1d' 'return_5d' 'return_10d' 'return_20d' 'sma_ratio_5' 'sma_ratio_10' 'sma_ratio_20'
 'sma_ratio_50' 'sma_ratio_200' 'ema_ratio_12' 'ema_ratio_26' 'macd_hist' 'rsi_14' 'bb_position'
 'volatility_10d' 'volatility_20d' 'volatility_60d' 'volume_sma_ratio_10' 'volume_sma_ratio_20'
 'daily_range' 'avg_range_10d' 'high_low_ratio' 'momentum_10d' 'momentum_20d' 'roc_5' 'roc_10'
 'atr_14']



,return_1d,return_5d,return_10d,return_20d,sma_ratio_5,sma_ratio_10,sma_ratio_20,sma_ratio_50,sma_ratio_200,ema_ratio_12,ema_ratio_26,macd_hist,rsi_14,bb_position,volatility_10d,volatility_20d,volatility_60d,volume_sma_ratio_10,volume_sma_ratio_20,daily_range,avg_range_10d,high_low_ratio,momentum_10d,momentum_20d,roc_5,roc_10,atr_14,stock_id
Train,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,nan
Test,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000


# **BASELINE MODELS**

We directly refit the 3 boosted tree models to the full dataset and make the test-set predictions. A VotingClassifier is helpful in this case as it automatically fits the base models and averages the results at the end.

In [4]:
estimators = [

    ("CB1C" , 
     CBC(
         iterations         = 1000,
         learning_rate      = 0.01,
         max_depth          = 4,
         objective          = "Logloss",
         eval_metric        = "AUC",
         verbose            = 0,
         l2_leaf_reg        = 0.70,
         colsample_bylevel  = 0.65,
         random_state       = 42,
     )
    ),
    
    ("XGB1C" , 
     XGBC(
         n_estimators       = 1000,
         learning_rate      = 0.01,
         max_depth          = 5,
         objective          = "binary:logistic",
         eval_metric        = "auc",
         colsample_bytree   = 0.55,
         verbosity          = 0,
         reg_lambda         = 2.50,
         reg_alpha          = 0.01,
         random_state       = 42,
         enable_categorical = True,
     )
    ),

    ("LGBM1C" , 
     LGBMC(
         n_estimators       = 1200,
         learning_rate      = 0.01,
         max_depth          = 5,
         objective          = "binary",
         eval_metric        = "auc",
         colsample_bytree   = 0.65,
         verbosity          = -1,
         reg_lambda         = 1.75,
         reg_alpha          = 0.10,
         random_state       = 42,
     )
    ), 

    ("LGBM2C" , 
     LGBMC(
         n_estimators       = 800,
         learning_rate      = 0.0125,
         max_depth          = 6,
         objective          = "binary",
         eval_metric        = "auc",
         data_sample_strategy = "goss",
         colsample_bytree   = 0.70,
         verbosity          = -1,
         reg_lambda         = 1.25,
         reg_alpha          = 0.10,
         random_state       = 42,
     )
    ),      
]

model = VotingClassifier(
    estimators = estimators,
    voting     = "soft",
    weights    = [0.40, 0.30, 0.30, 0.25],
    verbose    = True,
    flatten_transform = False,
)

X, y, Xtest = train[sel_cols], train[target], test[sel_cols]
model.fit(X, y)
test_preds = model.predict_proba(Xtest)[:,1]

sub_fl[target] = test_preds
sub_fl.to_csv("submission.csv", index = True)
print()
!head submission.csv

[Voting] ..................... (1 of 4) Processing CB1C, total=  48.3s
[Voting] .................... (2 of 4) Processing XGB1C, total=  23.7s
[Voting] ................... (3 of 4) Processing LGBM1C, total=  29.9s
[Voting] ................... (4 of 4) Processing LGBM2C, total=  29.8s

id,target
440402,0.505676759224385
440403,0.5426369474034752
440404,0.4991830106580008
440405,0.4850772565222198
440406,0.518963048110389
440407,0.5114030259279904
440408,0.5200744844183353
440409,0.5114935434120979
440410,0.5335510751621412
